# Thực hành Buổi 04+05 (bản gộp 30 phút): MapReduce trên YARN & Apache Spark — cụm Docker tự dựng

**Khóa đào tạo:** Phân tích Dữ liệu Lớn (Big Data Analysis)

**Thời lượng:** 30 phút | **Yêu cầu duy nhất:** Docker Desktop đang chạy (cấp ≥ 6 GB RAM)

> **Notebook này TỰ CHỨA 100%** — không gọi bất kỳ script hay hàm nào bên ngoài:
> - tự **sinh dữ liệu** (kho văn bản + 200.000 giao dịch, hạt giống cố định ➔ mọi máy ra số giống hệt nhau);
> - tự **viết `docker-compose.yml` và `hadoop.env`** rồi tự dựng cụm HDFS + YARN + Spark (tên container `lab45-*`, cổng riêng — không đụng cụm `bigdata-*` của bản lab đầy đủ);
> - mapper/reducer và script Spark đều **nhúng ngay trong các ô mã**.

| Phần | Nội dung | Thời gian |
|:---|:---|:---|
| 0 | Sinh dữ liệu ngay trong notebook | 4 phút |
| 1 | Tự viết compose + env, dựng cụm, nạp dữ liệu lên HDFS | 6 phút |
| 2 | **Buổi 04 rút gọn** — 2 job MapReduce Streaming thật trên YARN | 10 phút |
| 3 | **Buổi 05 rút gọn** — Spark trong cụm, cùng 2 phép tính, đọc thẳng HDFS | 7 phút |
| 4 | Bảng so sánh & tổng kết | 3 phút |

**Hai bài toán xuyên suốt** (trùng bản đầy đủ nên mọi con số đối chiếu được): WordCount — đáp án chuẩn **373 từ khác nhau / 1.144 tổng số từ**; Doanh thu theo ngành hàng — tổng **206.744.802.000 VND**.

## PHẦN 0 — SINH DỮ LIỆU NGAY TRONG NOTEBOOK (4 phút)

Không tải, không gọi script ngoài: kho văn bản nằm trong hằng `CORPUS` bên dưới; nhật ký giao dịch được sinh bằng `random.Random(42)` — **byte-by-byte giống hệt** dữ liệu của Buổi 04/05 đầy đủ, nên đáp án chuẩn dùng chung được.

In [9]:
import csv
import json
import random
import subprocess
import time
from datetime import datetime, timedelta
from pathlib import Path

THU_MUC_LAB = Path.cwd().resolve()                       # .../lab4_5
GOC = THU_MUC_LAB.parent if THU_MUC_LAB.name == "lab4_5" else THU_MUC_LAB
RAW = GOC / "data" / "raw";        RAW.mkdir(parents=True, exist_ok=True)
PROCESSED = GOC / "data" / "processed"; PROCESSED.mkdir(parents=True, exist_ok=True)
CORPUS_PATH = RAW / "wordcount_corpus.txt"
TX_PATH     = RAW / "transactions.csv"

# --- Đáp án chuẩn (trùng Buổi 04 và 05 đầy đủ) ---
CHUAN_WC = {"so_tu_khac_nhau": 373, "tong_so_tu": 1144}
CHUAN_REVENUE = {
    "Gia dụng":   (32231, 58925123000),
    "Điện tử":    (36296, 57872658000),
    "Thời trang": (27778, 47277119000),
    "Thực phẩm":  (59856, 18555159000),
    "Mỹ phẩm":    (19839, 17421590000),
    "Sách":       (24000,  6693153000),
}
TONG_DOANH_THU = 206_744_802_000
print("Gốc dự án:", GOC)

Gốc dự án: D:\DataAnalytics\Lab5


In [10]:
# --- Kho văn bản cho WordCount (nhúng trọn trong notebook) ---
CORPUS = """Dữ liệu lớn không phải là một công nghệ mà là một tình huống.
Tình huống ấy xảy ra khi dữ liệu vượt quá sức chứa của một máy tính duy nhất.
Khi dữ liệu vượt quá bộ nhớ của một máy, mọi thói quen xử lý cũ đều phải viết lại.

Một nhà máy hiện đại gắn cảm biến lên từng máy công cụ.
Mỗi cảm biến gửi về một bản ghi sau mỗi giây.
Một máy sinh ra tám mươi sáu nghìn bản ghi mỗi ngày.
Năm mươi máy sinh ra hơn bốn triệu bản ghi mỗi ngày.
Một năm vận hành liên tục sinh ra hàng tỉ bản ghi cảm biến.
Không một máy tính cá nhân nào giữ nổi khối dữ liệu đó trong bộ nhớ.

Hệ thống tệp phân tán ra đời để giải bài toán sức chứa.
Hệ thống tệp phân tán cắt một tệp lớn thành nhiều khối nhỏ.
Mỗi khối được đặt lên một máy khác nhau trong cụm.
Mỗi khối còn được nhân bản sang vài máy nữa để chống mất dữ liệu.
Khi một máy hỏng, khối dữ liệu vẫn còn nguyên trên những máy còn lại.
Cụm càng nhiều máy thì sức chứa của cụm càng lớn.

Hadoop gọi hệ thống tệp phân tán của mình là HDFS.
HDFS có một nút quản lý tên gọi NameNode.
NameNode giữ siêu dữ liệu: tệp nào gồm những khối nào, khối nào nằm ở máy nào.
NameNode không giữ dữ liệu, nó chỉ giữ bản đồ của dữ liệu.
Các nút lưu trữ tên gọi DataNode mới thực sự giữ từng khối.
DataNode báo cáo tình trạng khối của mình về NameNode theo chu kỳ.
Mất một DataNode thì cụm vẫn chạy, mất NameNode thì cụm mất bản đồ.

MapReduce là cách tính toán đi kèm với hệ thống tệp phân tán.
MapReduce chia phép tính thành hai hàm rất nhỏ.
Hàm map đọc từng dòng dữ liệu và phát ra một cặp khóa giá trị.
Hàm reduce nhận toàn bộ giá trị của cùng một khóa và tổng hợp lại.
Giữa map và reduce có một giai đoạn tên là shuffle.
Shuffle gom mọi cặp cùng khóa về cùng một máy.
Shuffle là giai đoạn tốn kém nhất vì nó phải truyền dữ liệu qua mạng.
Lập trình viên viết map và viết reduce, khung Hadoop lo phần shuffle.

Nguyên tắc quan trọng nhất của MapReduce tên là data locality.
Data locality nghĩa là đưa phép tính đến chỗ dữ liệu.
Chương trình chỉ nặng vài trăm kilobyte còn dữ liệu nặng hàng trăm gigabyte.
Chuyển chương trình qua mạng rẻ hơn chuyển dữ liệu qua mạng rất nhiều lần.
Mỗi máy tính trên chính khối dữ liệu nằm sẵn trên đĩa của mình.
Nhiều máy cùng tính một lúc nên tổng thời gian giảm xuống.

YARN là bộ quản lý tài nguyên của cụm Hadoop.
YARN quyết định job nào được cấp bao nhiêu bộ nhớ và bao nhiêu lõi.
ResourceManager của YARN nhận yêu cầu và phân bổ tài nguyên.
NodeManager của YARN chạy các tiến trình con trên từng máy.
Một cụm có thể chạy nhiều job cùng lúc nhờ YARN đứng ra điều phối.

WordCount là bài toán đầu tiên của mọi khóa học MapReduce.
WordCount đếm số lần xuất hiện của mỗi từ trong một kho văn bản.
Hàm map của WordCount tách dòng thành từ và phát ra cặp từ và số một.
Hàm reduce của WordCount cộng dồn tất cả số một của cùng một từ.
Bài toán đơn giản nhưng nó chứa đủ cả bốn giai đoạn của mô hình.
Đổi kho văn bản từ một megabyte thành một terabyte thì mã nguồn không đổi.
Chỉ có số máy trong cụm là phải đổi.

Nhật ký giao dịch của một chuỗi bán lẻ cũng là dữ liệu lớn.
Mỗi lần khách trả tiền, hệ thống ghi một dòng vào nhật ký.
Dòng nhật ký gồm mã giao dịch, thời gian, cửa hàng, ngành hàng, sản phẩm.
Dòng nhật ký còn ghi số lượng, đơn giá và phương thức thanh toán.
Doanh thu của một dòng bằng số lượng nhân đơn giá.
Tổng doanh thu theo ngành hàng chính là một phép reduce theo khóa ngành hàng.
Tổng doanh thu theo cửa hàng chính là một phép reduce theo khóa cửa hàng.
Cùng một dữ liệu, đổi khóa thì đổi báo cáo.

MapReduce ghi kết quả trung gian xuống đĩa sau mỗi giai đoạn.
Ghi xuống đĩa giúp job hồi phục được khi một máy chết giữa chừng.
Ghi xuống đĩa cũng khiến job chậm đi rất nhiều lần.
Một thuật toán lặp hai mươi vòng phải ghi và đọc đĩa hai mươi lần.
Đó là điểm yếu lớn nhất của MapReduce.

Apache Spark sinh ra để sửa đúng điểm yếu đó.
Spark giữ dữ liệu trung gian trong bộ nhớ thay vì ghi xuống đĩa.
Spark gọi tập dữ liệu phân tán của mình là RDD.
RDD là bất biến, đã tạo ra thì không sửa được nữa.
Mỗi phép biến đổi trên RDD sinh ra một RDD mới.
Spark ghi nhớ chuỗi biến đổi ấy dưới dạng một đồ thị.
Đồ thị đó cho phép Spark dựng lại phần dữ liệu bị mất mà không cần ghi đĩa.

Spark chia phép toán thành hai loại là biến đổi và hành động.
Biến đổi thì lười, Spark chỉ ghi lại chứ chưa tính.
Hành động mới kích hoạt toàn bộ chuỗi biến đổi đã ghi.
Nhờ lười mà Spark nhìn thấy trọn kế hoạch trước khi chạy.
Nhìn thấy trọn kế hoạch thì tối ưu được, gộp được nhiều bước làm một.

DataFrame là lớp trừu tượng cao hơn RDD.
DataFrame có lược đồ, biết tên cột và kiểu dữ liệu của từng cột.
Biết lược đồ thì bộ tối ưu Catalyst viết lại được câu truy vấn cho nhanh hơn.
Cùng một phép đếm, DataFrame thường nhanh hơn RDD viết tay.
Người mới nên bắt đầu từ DataFrame và chỉ dùng RDD khi thật cần.

So sánh công bằng giữa hai công nghệ đòi hỏi cùng một dữ liệu và cùng một phép tính.
Trên dữ liệu nhỏ, chi phí khởi động lấn át thời gian tính toán thật.
Trên dữ liệu nhỏ, một chương trình một máy luôn thắng cả Hadoop lẫn Spark.
Trên dữ liệu lớn, Hadoop và Spark mới cho thấy giá trị của mình.
Chọn công cụ theo quy mô dữ liệu chứ không theo độ mới của công nghệ.

Người làm dữ liệu cần biết cả hai.
Hadoop dạy ta cách nghĩ theo khóa và giá trị.
Spark cho ta cách nghĩ ấy với tốc độ của bộ nhớ.
Hiểu MapReduce thì hiểu luôn vì sao Spark nhanh.
Không hiểu MapReduce thì Spark chỉ còn là một thư viện lạ.
"""

if CORPUS_PATH.exists():
    print("wordcount_corpus.txt đã có — dùng lại.")
else:
    CORPUS_PATH.write_text(CORPUS, encoding="utf-8")
    print("Đã sinh wordcount_corpus.txt")
print(f"Kích thước: {CORPUS_PATH.stat().st_size/1024:,.1f} KB, "
      f"{len(CORPUS.splitlines())} dòng")

wordcount_corpus.txt đã có — dùng lại.
Kích thước: 6.9 KB, 102 dòng


In [11]:
# --- Nhật ký 200.000 giao dịch — sinh tuyến tính, hạt giống cố định seed=42 ---
if TX_PATH.exists():
    print("transactions.csv đã có — dùng lại.")
else:
    rng = random.Random(42)
    # (ngành hàng, [(tên sản phẩm, đơn giá cơ sở)...], trọng số xuất hiện)
    DANH_MUC = {
        "Điện tử": ([("Tai nghe không dây", 890_000), ("Chuột quang", 320_000),
                     ("Bàn phím cơ", 1_450_000), ("Ổ cứng SSD 512GB", 1_290_000),
                     ("Sạc dự phòng", 540_000)], 18),
        "Gia dụng": ([("Nồi cơm điện", 1_180_000), ("Ấm siêu tốc", 390_000),
                      ("Máy xay sinh tố", 760_000), ("Quạt điều hòa", 2_350_000),
                      ("Bộ dao nhà bếp", 450_000)], 16),
        "Thực phẩm": ([("Gạo ST25 5kg", 185_000), ("Dầu ăn 1L", 62_000),
                       ("Sữa tươi thùng", 340_000), ("Cà phê rang xay", 155_000),
                       ("Mì gói thùng", 128_000)], 30),
        "Thời trang": ([("Áo sơ mi", 420_000), ("Quần jeans", 680_000),
                        ("Giày thể thao", 1_250_000), ("Áo khoác gió", 890_000),
                        ("Túi xách", 1_540_000)], 14),
        "Sách": ([("Sách kỹ năng", 128_000), ("Sách thiếu nhi", 76_000),
                  ("Giáo trình đại học", 210_000), ("Truyện tranh", 45_000),
                  ("Từ điển", 320_000)], 12),
        "Mỹ phẩm": ([("Sữa rửa mặt", 245_000), ("Kem chống nắng", 385_000),
                     ("Son môi", 320_000), ("Nước hoa mini", 690_000),
                     ("Serum dưỡng da", 850_000)], 10),
    }
    CUA_HANG = [("S01", "Hà Nội"), ("S02", "Hà Nội"), ("S03", "Hải Phòng"),
                ("S04", "Đà Nẵng"), ("S05", "Huế"), ("S06", "TP HCM"),
                ("S07", "TP HCM"), ("S08", "TP HCM"), ("S09", "Cần Thơ"),
                ("S10", "Bình Dương")]
    TRONG_SO_CUA_HANG = [14, 11, 7, 9, 5, 16, 13, 10, 8, 7]
    THANH_TOAN = ["TIEN_MAT", "THE", "VI_DIEN_TU", "CHUYEN_KHOAN"]
    TRONG_SO_THANH_TOAN = [34, 30, 26, 10]

    ten_nganh = list(DANH_MUC)
    trong_so_nganh = [DANH_MUC[n][1] for n in ten_nganh]
    moc = datetime(2025, 1, 1, 7, 0, 0)

    with TX_PATH.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["transaction_id", "ts", "store_id", "city", "category",
                    "product", "quantity", "unit_price", "payment_method",
                    "customer_id"])
        for i in range(1, 200_001):
            moc += timedelta(seconds=rng.randint(1, 140))
            nganh = rng.choices(ten_nganh, weights=trong_so_nganh, k=1)[0]
            san_pham, gia_goc = rng.choice(DANH_MUC[nganh][0])
            don_gia = int(gia_goc * rng.uniform(0.92, 1.08) / 1000) * 1000
            so_luong = rng.choices([1, 2, 3, 4, 5], weights=[55, 24, 12, 6, 3], k=1)[0]
            cua_hang, thanh_pho = rng.choices(CUA_HANG, weights=TRONG_SO_CUA_HANG, k=1)[0]
            thanh_toan = rng.choices(THANH_TOAN, weights=TRONG_SO_THANH_TOAN, k=1)[0]
            khach = rng.randint(1, 25_000)
            w.writerow([f"T{i:07d}", moc.strftime("%Y-%m-%d %H:%M:%S"),
                        cua_hang, thanh_pho, nganh, san_pham, so_luong, don_gia,
                        thanh_toan, f"C{khach:05d}"])
    print("Đã sinh transactions.csv")
print(f"Kích thước: {TX_PATH.stat().st_size/1024**2:,.2f} MB")

transactions.csv đã có — dùng lại.
Kích thước: 18.46 MB


---
## PHẦN 1 — TỰ DỰNG CỤM HADOOP + SPARK BẰNG DOCKER (6 phút)

Ô dưới đây **viết ra hai tệp cấu hình** ngay trong thư mục `lab4_5/` rồi dựng cụm gồm 5 container:

| Container | Vai trò | Web UI |
|:---|:---|:---|
| `lab45-namenode` | HDFS NameNode — giữ *bản đồ* của dữ liệu | http://localhost:19870 |
| `lab45-datanode` | HDFS DataNode — thực sự giữ các khối dữ liệu | — |
| `lab45-resourcemanager` | YARN — cấp phát tài nguyên cho job | http://localhost:18088 |
| `lab45-nodemanager` | YARN — chạy tiến trình map/reduce trên "máy" của mình | — |
| `lab45-spark` | Spark — nơi ta `spark-submit` ở Phần 3 | http://localhost:18080 |

Tên container và cổng đều mang tiền tố riêng nên chạy song song được với cụm `bigdata-*` của bản lab đầy đủ. Quy ước của image `bde2020`: biến môi trường tiền tố `CORE_CONF_` / `HDFS_CONF_` / `YARN_CONF_` / `MAPRED_CONF_` được tự động chuyển thành thuộc tính trong `core-site.xml`, `hdfs-site.xml`, `yarn-site.xml`, `mapred-site.xml` (dấu `.` viết thành `_`, dấu `_` viết thành `___`).

In [12]:
# --- Tự viết hadoop.env và docker-compose.yml ---
HADOOP_ENV = """\
# Sinh tự động bởi Lab4_5.ipynb — cấu hình chung cho các container Hadoop
CORE_CONF_fs_defaultFS=hdfs://hadoop-namenode:9000
CORE_CONF_hadoop_http_staticuser_user=root
HDFS_CONF_dfs_replication=1
HDFS_CONF_dfs_webhdfs_enabled=true
HDFS_CONF_dfs_permissions_enabled=false
HDFS_CONF_dfs_namenode_datanode_registration_ip___hostname___check=false
YARN_CONF_yarn_resourcemanager_hostname=hadoop-resourcemanager
YARN_CONF_yarn_resourcemanager_address=hadoop-resourcemanager:8032
YARN_CONF_yarn_resourcemanager_scheduler_address=hadoop-resourcemanager:8030
YARN_CONF_yarn_resourcemanager_resource__tracker_address=hadoop-resourcemanager:8031
YARN_CONF_yarn_resourcemanager_webapp_address=hadoop-resourcemanager:8088
YARN_CONF_yarn_scheduler_capacity_root_default_maximum___allocation___mb=4096
YARN_CONF_yarn_scheduler_capacity_root_default_maximum___allocation___vcores=4
YARN_CONF_yarn_nodemanager_aux___services=mapreduce_shuffle
YARN_CONF_yarn_nodemanager_resource_memory___mb=3072
YARN_CONF_yarn_nodemanager_resource_cpu___vcores=4
YARN_CONF_yarn_nodemanager_vmem___check___enabled=false
YARN_CONF_yarn_nodemanager_disk___health___checker_max___disk___utilization___per___disk___percentage=99.0
YARN_CONF_yarn_timeline___service_enabled=false
YARN_CONF_yarn_log___aggregation___enable=false
MAPRED_CONF_mapreduce_framework_name=yarn
MAPRED_CONF_mapreduce_map_memory_mb=1024
MAPRED_CONF_mapreduce_reduce_memory_mb=1024
MAPRED_CONF_mapreduce_map_java_opts=-Xmx820m
MAPRED_CONF_mapreduce_reduce_java_opts=-Xmx820m
MAPRED_CONF_yarn_app_mapreduce_am_resource_mb=1024
MAPRED_CONF_yarn_app_mapreduce_am_command___opts=-Xmx820m
MAPRED_CONF_yarn_app_mapreduce_am_env=HADOOP_MAPRED_HOME=/opt/hadoop-3.2.1/
MAPRED_CONF_mapreduce_map_env=HADOOP_MAPRED_HOME=/opt/hadoop-3.2.1/
MAPRED_CONF_mapreduce_reduce_env=HADOOP_MAPRED_HOME=/opt/hadoop-3.2.1/
"""

DOCKER_COMPOSE = """\
# Sinh tự động bởi Lab4_5.ipynb — cụm tối thiểu cho buổi gộp 04+05
name: lab4-5-gop

networks:
  lab45-net:

volumes:
  lab45_namenode:
  lab45_datanode:

services:
  hadoop-namenode:
    image: bde2020/hadoop-namenode:2.0.0-hadoop3.2.1-java8
    container_name: lab45-namenode
    env_file: [./hadoop.env]
    environment:
      - CLUSTER_NAME=lab45_cluster
    ports:
      - "19870:9870"     # HDFS NameNode Web UI
    volumes:
      - lab45_namenode:/hadoop/dfs/name
    networks: [lab45-net]

  hadoop-datanode:
    image: bde2020/hadoop-datanode
    container_name: lab45-datanode
    env_file: [./hadoop.env]
    environment:
      - SERVICE_PRECONDITION=hadoop-namenode:9870
    volumes:
      - lab45_datanode:/hadoop/dfs/data
    depends_on: [hadoop-namenode]
    networks: [lab45-net]

  hadoop-resourcemanager:
    image: bde2020/hadoop-resourcemanager:2.0.0-hadoop3.2.1-java8
    container_name: lab45-resourcemanager
    env_file: [./hadoop.env]
    environment:
      - SERVICE_PRECONDITION=hadoop-namenode:9870 hadoop-datanode:9864
    ports:
      - "18088:8088"     # YARN ResourceManager Web UI
    depends_on: [hadoop-datanode]
    networks: [lab45-net]

  hadoop-nodemanager:
    image: bde2020/hadoop-nodemanager:2.0.0-hadoop3.2.1-java8
    container_name: lab45-nodemanager
    env_file: [./hadoop.env]
    environment:
      - SERVICE_PRECONDITION=hadoop-namenode:9870 hadoop-datanode:9864 hadoop-resourcemanager:8088
    depends_on: [hadoop-resourcemanager]
    networks: [lab45-net]

  spark-master:
    image: bitnamilegacy/spark:3.5.1
    container_name: lab45-spark
    environment:
      - SPARK_MODE=master
      - SPARK_RPC_AUTHENTICATION_ENABLED=no
    ports:
      - "18080:8080"     # Spark Master Web UI
    networks: [lab45-net]
"""

(THU_MUC_LAB / "hadoop.env").write_text(HADOOP_ENV, encoding="utf-8")
(THU_MUC_LAB / "docker-compose.yml").write_text(DOCKER_COMPOSE, encoding="utf-8")
print("Đã viết lab4_5/hadoop.env       (", len(HADOOP_ENV.splitlines()), "dòng )")
print("Đã viết lab4_5/docker-compose.yml (", len(DOCKER_COMPOSE.splitlines()), "dòng )")

Đã viết lab4_5/hadoop.env       ( 31 dòng )
Đã viết lab4_5/docker-compose.yml ( 63 dòng )


In [13]:
# --- Dựng cụm và chờ sẵn sàng ---
kt = subprocess.run(["docker", "info"], capture_output=True, text=True)
if kt.returncode != 0:
    raise RuntimeError("Docker chưa chạy — hãy mở Docker Desktop rồi chạy lại ô này.")

print("docker compose up -d ... (lần đầu có thể tải image, vài phút)")
subprocess.run(["docker", "compose", "up", "-d"], cwd=THU_MUC_LAB, check=True)

print("Chờ cụm sẵn sàng (HDFS rời safemode, DataNode và NodeManager đăng ký)...")
t_bat_dau = time.time()
san_sang = False
while time.time() - t_bat_dau < 300:
    safemode = subprocess.run(
        ["docker", "exec", "lab45-namenode", "hdfs", "dfsadmin", "-safemode", "get"],
        capture_output=True, text=True).stdout
    bao_cao = subprocess.run(
        ["docker", "exec", "lab45-namenode", "hdfs", "dfsadmin", "-report"],
        capture_output=True, text=True).stdout
    nut_yarn = subprocess.run(
        ["docker", "exec", "lab45-resourcemanager", "yarn", "node", "-list"],
        capture_output=True, text=True).stdout
    if ("Safe mode is OFF" in safemode and "Live datanodes (1)" in bao_cao
            and "RUNNING" in nut_yarn):
        san_sang = True
        break
    print(f"   ... {int(time.time() - t_bat_dau):>3}s")
    time.sleep(10)

if not san_sang:
    raise RuntimeError("Cụm chưa sẵn sàng sau 5 phút — xem log: docker compose logs")
print(f"CỤM SẴN SÀNG sau {time.time() - t_bat_dau:.0f} giây.")
print("HDFS UI: http://localhost:19870 | YARN UI: http://localhost:18088")

docker compose up -d ... (lần đầu có thể tải image, vài phút)
Chờ cụm sẵn sàng (HDFS rời safemode, DataNode và NodeManager đăng ký)...
CỤM SẴN SÀNG sau 4 giây.
HDFS UI: http://localhost:19870 | YARN UI: http://localhost:18088


In [14]:
# --- Nạp dữ liệu lên HDFS ---
HDFS_IN  = "/user/bigdata/lab45/input"
HDFS_OUT = "/user/bigdata/lab45/output"

subprocess.run(["docker", "cp", str(CORPUS_PATH), "lab45-namenode:/tmp/"], check=True)
subprocess.run(["docker", "cp", str(TX_PATH), "lab45-namenode:/tmp/"], check=True)
subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c",
                f"hdfs dfs -mkdir -p {HDFS_IN}"
                f" && hdfs dfs -put -f /tmp/wordcount_corpus.txt /tmp/transactions.csv {HDFS_IN}/"
                f" && hdfs dfs -rm -r -f -skipTrash {HDFS_OUT}"], check=True,
               capture_output=True, text=True)
ds = subprocess.run(["docker", "exec", "lab45-namenode", "hdfs", "dfs", "-ls", "-h", HDFS_IN],
                    capture_output=True, text=True).stdout
print("Dữ liệu trên HDFS:")
print(ds)

Dữ liệu trên HDFS:
Found 2 items
-rw-r--r--   1 root supergroup     18.5 M 2026-08-27 04:33 /user/bigdata/lab45/input/transactions.csv
-rw-r--r--   1 root supergroup      6.9 K 2026-08-27 04:33 /user/bigdata/lab45/input/wordcount_corpus.txt



---
## PHẦN 2 — BUỔI 04 RÚT GỌN: JOB MAPREDUCE THẬT TRÊN YARN (10 phút)

MapReduce chia mọi phép tính lớn thành bốn giai đoạn:

| Giai đoạn | Ai làm | Việc làm |
|:---|:---|:---|
| **1. Split** | khung Hadoop | cắt tệp thành khối, phát cho các máy |
| **2. Map** | *bạn viết* | đọc từng dòng ➔ phát ra cặp `khóa <TAB> giá_trị` |
| **3. Shuffle & Sort** | khung Hadoop | gom mọi cặp **cùng khóa** về cùng một máy — tốn kém nhất vì đi qua mạng |
| **4. Reduce** | *bạn viết* | nhận các giá trị của một khóa (đã sắp xếp liền nhau) ➔ tổng hợp |

**Hợp đồng của Hadoop Streaming:** mapper/reducer là *bất kỳ chương trình nào* đọc stdin, ghi stdout. Container Hadoop của khóa học không cài Python, nên ta viết bằng **awk** — bốn script nhúng ngay bên dưới. Chú ý reducer chỉ cần **một biến đếm** (không cần dictionary) vì Hadoop bảo đảm dòng vào đã sắp xếp theo khóa — nhờ vậy reducer xử lý được luồng dữ liệu lớn hơn RAM.

In [15]:
# --- 4 script mapper/reducer nhúng trong notebook ---
WC_MAPPER = """#!/bin/bash
# MAPPER WordCount: dòng văn bản -> "tu <TAB> 1"
# tolower() của awk chỉ hạ chữ hoa ASCII, giữ nguyên Đ / Ổ / Ầ (khớp đáp án chuẩn)
awk '{
    gsub(/\\r/, "")
    gsub(/[.,;:!?\"()\\[\\]]/, " ")
    n = split(tolower($0), w, /[ \\t]+/)
    for (i = 1; i <= n; i++) if (w[i] != "") print w[i] "\\t1"
}'
"""

WC_REDUCER = """#!/bin/bash
# REDUCER WordCount: stdin ĐÃ SẮP XẾP theo khóa -> "tu <TAB> tổng"
awk -F'\\t' '
$1 != khoa { if (khoa != "") print khoa "\\t" tong; khoa = $1; tong = 0 }
           { tong += $2 }
END        { if (khoa != "") print khoa "\\t" tong }
'
"""

TX_MAPPER = """#!/bin/bash
# MAPPER Doanh thu: cột 5 = category, 7 = quantity, 8 = unit_price
awk -F',' '$1 != "transaction_id" { print $5 "\\t" $7 * $8 }'
"""

TX_REDUCER = """#!/bin/bash
# REDUCER Doanh thu -> "category <TAB> so_giao_dich <TAB> doanh_thu"
awk -F'\\t' '
$1 != khoa { if (khoa != "") printf "%s\\t%d\\t%.0f\\n", khoa, dem, tong; khoa = $1; dem = 0; tong = 0 }
           { dem += 1; tong += $2 }
END        { if (khoa != "") printf "%s\\t%d\\t%.0f\\n", khoa, dem, tong }
'
"""

import tempfile
tam = Path(tempfile.mkdtemp())
for ten, noi_dung in [("wc_mapper.sh", WC_MAPPER), ("wc_reducer.sh", WC_REDUCER),
                      ("tx_mapper.sh", TX_MAPPER), ("tx_reducer.sh", TX_REDUCER)]:
    (tam / ten).write_text(noi_dung, encoding="utf-8", newline="\n")
subprocess.run(["docker", "exec", "lab45-namenode", "mkdir", "-p", "/tmp/mr"], check=True)
subprocess.run(["docker", "cp", f"{tam}/.", "lab45-namenode:/tmp/mr/"], check=True)
subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c", "chmod +x /tmp/mr/*.sh"],
               check=True)
print("Đã đưa 4 script vào lab45-namenode:/tmp/mr/")


Đã đưa 4 script vào lab45-namenode:/tmp/mr/


**Kiểm thử bằng ống dẫn Unix trước khi nộp job** — `sort` chính là Shuffle & Sort thu nhỏ. Vài giây thay vì ~30 giây, và lỗi (nếu có) đọc được ngay thay vì chôn trong log YARN:

In [16]:
kq = subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c",
    f"hdfs dfs -cat {HDFS_IN}/wordcount_corpus.txt 2>/dev/null"
    " | /tmp/mr/wc_mapper.sh | LC_ALL=C sort | /tmp/mr/wc_reducer.sh"],
    capture_output=True, text=True, encoding="utf-8").stdout

dong = [d for d in kq.strip().split("\n") if d]
print(f"Ống dẫn Unix: {len(dong)} từ khác nhau, "
      f"{sum(int(d.split(chr(9))[1]) for d in dong):,} tổng số từ")
assert len(dong) == CHUAN_WC["so_tu_khac_nhau"]
print("==> KHỚP đáp án chuẩn — yên tâm nộp job lên YARN")


Ống dẫn Unix: 373 từ khác nhau, 1,144 tổng số từ
==> KHỚP đáp án chuẩn — yên tâm nộp job lên YARN


In [19]:
# --- JOB 2 TRÊN YARN: Doanh thu theo ngành hàng ---
# 1. Xóa thư mục output cũ của Job 2 nếu có
subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c",
                f"hdfs dfs -rm -r -f -skipTrash {HDFS_OUT}/revenue"], capture_output=True)

# 2. Nộp job Doanh thu lên YARN
t0 = time.perf_counter()
job = subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c",
    f"hadoop jar {STREAM_JAR}"
    " -D mapreduce.job.name=Lab45_Revenue -D mapreduce.job.reduces=1"
    " -files /tmp/mr/tx_mapper.sh,/tmp/mr/tx_reducer.sh"
    " -mapper tx_mapper.sh -reducer tx_reducer.sh"
    f" -input {HDFS_IN}/transactions.csv -output {HDFS_OUT}/revenue 2>&1"],
    capture_output=True, text=True)
t_yarn_tx = time.perf_counter() - t0

kq = subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c",
    f"hdfs dfs -cat {HDFS_OUT}/revenue/part-*"], capture_output=True, text=True).stdout
ket_qua_mr = {}
for d in kq.strip().split("\n"):
    p = d.split(chr(9))
    if len(p) == 3:
        ket_qua_mr[p[0]] = (int(p[1]), int(p[2]))
for nganh, (so_gd, dt) in sorted(ket_qua_mr.items(), key=lambda x: -x[1][1]):
    print(f"  {nganh:<12}{so_gd:>9,}{dt:>18,}")
print(f"Thời gian job Doanh thu trên YARN: {t_yarn_tx:.1f} s")

assert ket_qua_mr == CHUAN_REVENUE
assert sum(v[1] for v in ket_qua_mr.values()) == TONG_DOANH_THU
print(f"==> KHỚP đáp án chuẩn — tổng {TONG_DOANH_THU:,} VND")


Exception in thread Thread-45 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 1052, in _bootstrap_inner
    self.run()
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\threading.py", line 989, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1597, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 86: character maps to <undefined>


AttributeError: 'NoneType' object has no attribute 'strip'

In [ ]:
# --- JOB 2 TRÊN YARN: Doanh thu theo ngành hàng (đổi khóa là đổi báo cáo) ---
t0 = time.perf_counter()
job = subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c",
    f"hadoop jar {STREAM_JAR}"
    " -D mapreduce.job.name=Lab45_Revenue -D mapreduce.job.reduces=1"
    " -files /tmp/mr/tx_mapper.sh,/tmp/mr/tx_reducer.sh"
    " -mapper tx_mapper.sh -reducer tx_reducer.sh"
    f" -input {HDFS_IN}/transactions.csv -output {HDFS_OUT}/revenue 2>&1"],
    capture_output=True, text=True)
t_yarn_tx = time.perf_counter() - t0

kq = subprocess.run(["docker", "exec", "lab45-namenode", "bash", "-c",
    f"hdfs dfs -cat {HDFS_OUT}/revenue/part-*"], capture_output=True, text=True).stdout
ket_qua_mr = {}
for d in kq.strip().split("\n"):
    p = d.split(chr(9))
    if len(p) == 3:
        ket_qua_mr[p[0]] = (int(p[1]), int(p[2]))
for nganh, (so_gd, dt) in sorted(ket_qua_mr.items(), key=lambda x: -x[1][1]):
    print(f"  {nganh:<12}{so_gd:>9,}{dt:>18,}")
print(f"Thời gian job Doanh thu trên YARN: {t_yarn_tx:.1f} s")

assert ket_qua_mr == CHUAN_REVENUE
assert sum(v[1] for v in ket_qua_mr.values()) == TONG_DOANH_THU
print(f"==> KHỚP đáp án chuẩn — tổng {TONG_DOANH_THU:,} VND")

> **Đọc con số cho đúng:** mỗi job mất ~25–40 giây dù phép tính thật chưa đến 1 giây — vì dữ liệu quá nhỏ, gần như toàn bộ thời gian là **chi phí cố định**: xin tài nguyên YARN, khởi động JVM cho từng task, và **ghi kết quả trung gian xuống đĩa** sau mỗi giai đoạn (cái giá của khả năng chịu lỗi). Và mỗi job **trả lại chi phí đó từ đầu**. Đây chính là điểm yếu mà Spark sinh ra để sửa.

---
## PHẦN 3 — BUỔI 05 RÚT GỌN: SPARK TRONG CỤM, ĐỌC THẲNG HDFS (7 phút)

Spark giữ dữ liệu trung gian **trong RAM** và khởi động JVM **một lần cho cả phiên**. Script dưới đây (nhúng trong biến `SCRIPT_SPARK`) chạy **cả hai phép tính trong một phiên** — cùng phần cứng, cùng lớp giả lập, cùng dữ liệu HDFS với hai job MapReduce vừa rồi, nên phép so sánh là **công bằng**. `reduceByKey` của Spark cộng dồn ngay tại mỗi máy trước khi gửi qua mạng — chính là "combiner" của Hadoop, nhưng tự động.

In [ ]:
SCRIPT_SPARK = r"""
import time
from pyspark.sql import SparkSession, functions as F

HDFS = "hdfs://hadoop-namenode:9000/user/bigdata/lab45/input"

t_khoi_dong = time.perf_counter()
spark = (SparkSession.builder.appName("Lab45_Gop")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
t_khoi_dong = time.perf_counter() - t_khoi_dong

# Bài 1: doanh thu theo ngành hàng (DataFrame API)
t0 = time.perf_counter()
df = spark.read.csv(HDFS + "/transactions.csv", header=True, inferSchema=True)
ket_qua = (df.withColumn("revenue", F.col("quantity") * F.col("unit_price"))
             .groupBy("category")
             .agg(F.count("*").alias("so_gd"), F.sum("revenue").alias("doanh_thu"))
             .orderBy(F.desc("doanh_thu")).collect())
t_revenue = time.perf_counter() - t0

# Bài 2: WordCount (RDD API — flatMap + reduceByKey)
DAU_CAU  = str.maketrans({c: " " for c in '.,;:!?"()[]'})
HA_ASCII = str.maketrans("ABCDEFGHIJKLMNOPQRSTUVWXYZ", "abcdefghijklmnopqrstuvwxyz")
t0 = time.perf_counter()
wc = (spark.sparkContext.textFile(HDFS + "/wordcount_corpus.txt")
      .flatMap(lambda d: d.translate(DAU_CAU).translate(HA_ASCII).split())
      .map(lambda tu: (tu, 1)).reduceByKey(lambda a, b: a + b))
so_tu = wc.count()
tong_luot = wc.map(lambda x: x[1]).sum()
t_wordcount = time.perf_counter() - t0

print("KQ|khoi_dong|%.2f" % t_khoi_dong)
print("KQ|revenue_s|%.2f" % t_revenue)
print("KQ|wordcount_s|%.2f" % t_wordcount)
for r in ket_qua:
    print("KQ|nganh|%s|%d|%d" % (r["category"], r["so_gd"], r["doanh_thu"]))
print("KQ|so_tu|%d|%d" % (so_tu, int(tong_luot)))
spark.stop()
"""

(tam / "spark_gop.py").write_text(SCRIPT_SPARK, encoding="utf-8")
subprocess.run(["docker", "cp", str(tam / "spark_gop.py"), "lab45-spark:/tmp/"], check=True)

print("ĐANG CHẠY spark-submit trong container (10–30 giây)...")
t0 = time.perf_counter()
kq = subprocess.run(["docker", "exec", "lab45-spark",
                     "/opt/bitnami/spark/bin/spark-submit",
                     "--master", "local[*]", "/tmp/spark_gop.py"],
                    capture_output=True, text=True)
t_spark_tong = time.perf_counter() - t0

so_lieu, ket_qua_spark, wc_spark = {}, {}, None
for d in (kq.stdout + kq.stderr).split("\n"):
    if d.startswith("KQ|"):
        p = d.split("|")
        if p[1] == "nganh":
            ket_qua_spark[p[2]] = (int(p[3]), int(p[4]))
        elif p[1] == "so_tu":
            wc_spark = (int(p[2]), int(p[3]))
        else:
            so_lieu[p[1]] = float(p[2])

print(f"Khởi tạo SparkSession : {so_lieu['khoi_dong']:6.2f} s   (trả MỘT lần)")
print(f"Doanh thu theo ngành  : {so_lieu['revenue_s']:6.2f} s   (job YARN mất ~{t_yarn_tx:.0f} s)")
print(f"WordCount             : {so_lieu['wordcount_s']:6.2f} s   (job YARN mất ~{t_yarn_wc:.0f} s)")
for nganh, (so_gd, dt) in sorted(ket_qua_spark.items(), key=lambda x: -x[1][1]):
    print(f"  {nganh:<12}{so_gd:>9,}{dt:>18,}")
print(f"Số từ khác nhau: {wc_spark[0]} | tổng lượt: {wc_spark[1]:,}")
print(f"Tổng thời gian spark-submit (kể cả khởi động JVM): {t_spark_tong:.1f} s")

assert ket_qua_spark == CHUAN_REVENUE and wc_spark == (373, 1144)
print("\n==> Spark cho KẾT QUẢ GIỐNG HỆT hai job MapReduce — chỉ nhanh hơn, không tính khác")

---
## PHẦN 4 — BẢNG SO SÁNH CÔNG BẰNG & TỔNG KẾT (3 phút)

In [ ]:
print(f"{'Phép tính':<26}{'MapReduce trên YARN':>22}{'Spark trong cụm':>18}")
print("-" * 66)
print(f"{'WordCount (7 KB)':<26}{t_yarn_wc:>20.1f} s{so_lieu['wordcount_s']:>16.2f} s")
print(f"{'Doanh thu (18,5 MB)':<26}{t_yarn_tx:>20.1f} s{so_lieu['revenue_s']:>16.2f} s")
print(f"{'Tổng (kể cả khởi động)':<26}{t_yarn_wc + t_yarn_tx:>20.1f} s{t_spark_tong:>16.1f} s")

print("""
BA ĐIỀU RÚT RA (so sánh CÔNG BẰNG: cùng máy, cùng container, cùng dữ liệu HDFS):
1. Kết quả GIỐNG HỆT tới từng con số — Spark không tính khác,
   nó chỉ KHÔNG ghi đĩa giữa các giai đoạn.
2. Hai phép tính chạy trong MỘT phiên Spark; MapReduce cần HAI job,
   mỗi job trả lại chi phí khởi động từ đầu.
3. Dữ liệu càng nhỏ, chi phí cố định càng lấn át — chọn công cụ theo
   QUY MÔ DỮ LIỆU, không theo độ mới của công nghệ.
""")

ket_qua_json = {
    "t_yarn_wordcount_s": round(t_yarn_wc, 1),
    "t_yarn_revenue_s":   round(t_yarn_tx, 1),
    "t_spark_tong_s":     round(t_spark_tong, 1),
    "t_spark_wordcount_s": so_lieu["wordcount_s"],
    "t_spark_revenue_s":   so_lieu["revenue_s"],
    "wordcount_khop": True, "revenue_khop": True,
}
(PROCESSED / "lab4_5_ket_qua.json").write_text(
    json.dumps(ket_qua_json, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã ghi", PROCESSED / "lab4_5_ket_qua.json")

## Dọn dẹp (chạy khi học xong — cụm chiếm ~4–5 GB RAM)

Mở terminal trong thư mục `lab4_5/` và chạy:

```bash
docker compose down -v
```

(`-v` xóa luôn dữ liệu HDFS của cụm; hai tệp `docker-compose.yml`, `hadoop.env` do notebook sinh ra — chạy lại PHẦN 1 là dựng lại từ đầu.)

---
# BÀI TẬP VỀ NHÀ

Làm trực tiếp vào các cell trống bên dưới. Nộp notebook đã chạy đầy đủ output.

#### Bài tập 1 (MapReduce): Viết cặp mapper/reducer awk tính **doanh thu theo thành phố** (cột 4 — `city`), nộp thành job thứ ba trên YARN theo đúng khuôn của Phần 2. Thành phố nào đứng đầu?

In [ ]:
# --- BÀI TẬP 1 ---


#### Bài tập 2 (Spark): Sửa `SCRIPT_SPARK` để tính thêm doanh thu theo thành phố (thêm một `groupBy` trong CÙNG phiên), chạy lại `spark-submit` và `assert` kết quả khớp với job MapReduce ở bài tập 1. Tổng thời gian tăng thêm bao nhiêu — và vì sao ít hơn hẳn một job YARN mới?

In [ ]:
# --- BÀI TẬP 2 ---
